# 🚦 RoadSign Evaluator — Generador de Modelo IA

Este cuaderno descarga un modelo YOLOv8 entrenado para detectar señales de tráfico, lo convierte al formato ONNX que usa tu app y te da el archivo listo para subir a GitHub.

**Cómo usarlo (todo automático):**
1. Arriba en el menú: **Entorno de ejecución → Ejecutar todo** (o `Ctrl+F9`)
2. Espera 2-3 minutos
3. Al final se descargan automáticamente `model.onnx` y `labels.json`
4. Sube esos 2 archivos a la carpeta `models/` de tu repositorio

No necesitas GPU ni saber programar. Solo ejecutar.

## Paso 1 — Instalar herramientas

In [ ]:
%pip install -q ultralytics onnx onnxslim
print('✅ Herramientas instaladas')

## Paso 2 — Descargar modelo de señales de tráfico

Usamos un modelo YOLOv8 pre-entrenado con el dataset GTSDB (señales europeas, válidas para España). Si la descarga del modelo especializado falla, el cuaderno usa YOLOv8n base como respaldo y lo deja listo igualmente.

In [ ]:
import urllib.request, os
from ultralytics import YOLO

MODEL_PT = 'traffic_signs.pt'
got_specialized = False

# Intento 1: modelo especializado en señales (Hugging Face / repos públicos)
SOURCES = [
    # Modelo YOLOv8 entrenado en señales de tráfico (GTSDB-like)
    'https://huggingface.co/keremberke/yolov8n-traffic-signs/resolve/main/best.pt',
]

for url in SOURCES:
    try:
        print(f'Descargando modelo especializado…')
        urllib.request.urlretrieve(url, MODEL_PT)
        if os.path.getsize(MODEL_PT) > 1_000_000:  # >1MB = válido
            got_specialized = True
            print('✅ Modelo especializado de señales descargado')
            break
    except Exception as e:
        print(f'  No disponible: {e}')

# Respaldo: YOLOv8n base (detecta stop sign, traffic light)
if not got_specialized:
    print('Usando YOLOv8n base como respaldo…')
    MODEL_PT = 'yolov8n.pt'
    YOLO(MODEL_PT)  # descarga automática
    print('✅ YOLOv8n base listo')

print(f'\nModelo a usar: {MODEL_PT}')

## Paso 3 — Convertir a ONNX

Exportamos a ONNX con tamaño de entrada 640×640 y optimización para que sea ligero y rápido en el móvil.

In [ ]:
from ultralytics import YOLO
import json

model = YOLO(MODEL_PT)

# Exportar a ONNX (opset 12 = máxima compatibilidad con ONNX Runtime Web)
onnx_path = model.export(
    format='onnx',
    imgsz=640,
    opset=12,
    simplify=True,
    dynamic=False
)
print(f'✅ Modelo ONNX generado: {onnx_path}')

# Renombrar a model.onnx
import shutil
shutil.move(onnx_path, 'model.onnx')
print('✅ Renombrado a model.onnx')

# Extraer las etiquetas de clase del modelo
names = model.names  # dict {0:'nombre', 1:'nombre', ...}
labels = [names[i] for i in range(len(names))]
with open('labels.json', 'w', encoding='utf-8') as f:
    json.dump(labels, f, ensure_ascii=False, indent=2)
print(f'✅ {len(labels)} clases guardadas en labels.json')
print('\nClases del modelo:')
for i, n in enumerate(labels):
    print(f'  {i}: {n}')

## Paso 4 — Verificar el tamaño

Si el modelo pesa más de ~15MB puede ir lento en móviles antiguos. YOLOv8n suele quedar en 12-13MB, perfecto.

In [ ]:
import os
size_mb = os.path.getsize('model.onnx') / 1024 / 1024
print(f'📦 Tamaño de model.onnx: {size_mb:.1f} MB')
if size_mb < 16:
    print('✅ Tamaño óptimo para móvil')
else:
    print('⚠️ Algo grande; funcionará pero puede tardar en cargar la 1ª vez')

## Paso 5 — Descargar los archivos

Se descargan `model.onnx` y `labels.json`. **Súbelos a la carpeta `models/` de tu repositorio de GitHub.**

In [ ]:
from google.colab import files
print('Descargando model.onnx…')
files.download('model.onnx')
print('Descargando labels.json…')
files.download('labels.json')
print('\n✅ ¡Listo! Sube ambos archivos a la carpeta models/ de tu repo.')